# Triton Puzzles — Part 3: Hard (online softmax, flash-attention, atomics, swizzle)

**Puzzles 22–26.** The good stuff. By the end of this notebook you will have written, from scratch, the algorithmic core of FlashAttention.

Assumes Parts 1 and 2 — you need to be comfortable with 2D tile pointers, `tl.dot`, and block reductions.


## 0. The GPU mental model (read this once)

Before Triton makes sense, you need a picture of the machine.

```
         GPU
 ┌──────────────────────────────────────────────┐
 │  SM 0   SM 1   SM 2  ...  SM N                │   <- ~80–140 SMs on modern GPUs
 │  ┌──┐  ┌──┐                                   │
 │  │  │  │  │   each SM runs many WARPS (32     │
 │  │  │  │  │   threads in lockstep, SIMT)       │
 │  └──┘  └──┘                                   │
 │   |     |                                     │
 │   shared mem / L1   (~100KB per SM, fast)     │
 └──┬──────┬─────────────────────────────────────┘
    │      │
    └──────┴──── L2 cache (tens of MB, shared)
               │
               └── HBM / global memory (slow, GB)
```

**Triton's bargain with you**: you don't write per-thread code (like CUDA), you write per-*program* code. A program ≈ a CUDA block ≈ a tile of work. Inside, you operate on *vectors* (`BLOCK_SIZE` elements). The compiler vectorizes across threads, picks register allocation, decides shared-memory staging, and does software pipelining.

Key Triton primitives you'll use everywhere:
- `pid = tl.program_id(axis=0)` — which tile am I?
- `offs = pid * BLOCK + tl.arange(0, BLOCK)` — the row of indices this tile owns.
- `mask = offs < N` — guard for the tail tile.
- `x = tl.load(ptr + offs, mask=mask, other=0.0)` — vectorized gather from HBM.
- `tl.store(ptr + offs, val, mask=mask)` — vectorized scatter.

🧠 **Fun fact**: `tl.arange(0, BLOCK)` must have a `BLOCK` that is a *power of two known at compile time*. That's because Triton lowers the vector to a fixed-shape MLIR tensor; the compiler needs the shape to pick layouts. The same is why `BLOCK_SIZE` is a `tl.constexpr`.

In [ ]:
import os, math, time
import torch
import triton
import triton.language as tl

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    os.environ['TRITON_INTERPRET'] = '1'   # let kernels run on CPU for learning
    print('No GPU detected — using Triton interpreter mode. Slow but debuggable.')
else:
    print('Device:', torch.cuda.get_device_name(0))
    print('SMs   :', torch.cuda.get_device_properties(0).multi_processor_count)

torch.manual_seed(0)

def check(out, ref, atol=1e-3, rtol=1e-3, name=''):
    ok = torch.allclose(out, ref, atol=atol, rtol=rtol)
    diff = (out - ref).abs().max().item()
    print(f"{'✅' if ok else '❌'} {name}  max|Δ|={diff:.3e}")
    return ok


### A quick visualization of how `program_id` tiles a vector

```
 vector length N = 13,  BLOCK = 4   →  grid = ceil(13/4) = 4 programs

  index : 0  1  2  3 | 4  5  6  7 | 8  9 10 11 |12  X  X  X
  pid   :     0      |     1      |     2      |     3 (tail, masked)
```

Every program independently computes its slice. There is **no implicit communication between programs** — if you want a global reduction across tiles, you either do a second kernel or use atomics.

## Puzzle 22 — 1D convolution (small kernel)
`out[i] = sum_k x[i+k] * w[k]` for `k in [0, K)`. Each program owns a contiguous output tile. Reads of `x` overlap between neighbors — that's the *halo* in stencils.

In [ ]:
@triton.jit
def k_conv1d(x_ptr,w_ptr,y_ptr,N,K,BLOCK: tl.constexpr,KSIZE: tl.constexpr):
    pid=tl.program_id(0)
    rn = pid*BLOCK + tl.arange(0,BLOCK)              # output positions
    acc = tl.zeros((BLOCK,), dtype=tl.float32)
    # TODO: for k in tl.static_range(KSIZE):
    #          x = tl.load(x_ptr + rn + k, mask=(rn+k) < N, other=0.0)
    #          wk = tl.load(w_ptr + k)
    #          acc += x * wk
    # TODO: store acc to y_ptr+rn, mask rn<(N-K+1)
    pass

def test_p22():
    N=10_000; K=5
    x=torch.randn(N,device=DEVICE); w=torch.randn(K,device=DEVICE)
    out_len = N - K + 1; y=torch.empty(out_len,device=DEVICE)
    k_conv1d[(triton.cdiv(out_len,256),)](x,w,y,N,K,BLOCK=256,KSIZE=K)
    ref = torch.nn.functional.conv1d(x.view(1,1,-1), w.flip(0).view(1,1,-1)).view(-1)
    check(y, ref, atol=1e-3, name='P22 conv1d')
test_p22()


## Puzzle 23 — Online (streaming) softmax

Now the row is **too big to fit in one tile** (e.g. attention scores of length 16384). You need to compute softmax in one pass over chunks, maintaining a running max `m` and normalizer `l`. The recurrence (Milakov & Gimelshein, 2018):

```
  start:  m = -inf,  l = 0
  for each chunk x_b:
      m_new = max(m, max(x_b))
      l     = l * exp(m - m_new) + sum(exp(x_b - m_new))
      m     = m_new
  finalize: y_b = exp(x_b - m) / l    # second pass, or carry exponentials
```

This is the **exact recurrence at the heart of FlashAttention**. Implement it for a single 1D vector of length `N`, in one program, looping over chunks of size `BLOCK`.

In [ ]:
@triton.jit
def k_online_softmax(x_ptr,y_ptr,N,BLOCK: tl.constexpr):
    # pass 1: compute m, l
    m = -float('inf')
    l = 0.0
    # TODO: loop over chunks; update m, l with the recurrence above
    # for k in range(0, N, BLOCK):
    #     rk = k + tl.arange(0, BLOCK); mask = rk < N
    #     x = tl.load(x_ptr+rk, mask=mask, other=-float('inf'))
    #     m_new = tl.maximum(m, tl.max(x, 0))
    #     l = l * tl.exp(m - m_new) + tl.sum(tl.exp(x - m_new), 0)
    #     m = m_new

    # pass 2: write y = exp(x - m) / l
    # TODO: same loop, but compute y and store
    pass

def test_p23():
    N=12_345; x=torch.randn(N,device=DEVICE); y=torch.empty_like(x)
    k_online_softmax[(1,)](x,y,N,BLOCK=1024)
    check(y, torch.softmax(x,0), atol=1e-5, name='P23 online_softmax')
test_p23()


## Puzzle 24 — Flash-attention-lite (1 head, fp32, causal=False)

Putting it all together. Inputs `Q,K,V` of shape `(L, D)` (single head, single batch). Compute `O = softmax(Q @ K.T / sqrt(D)) @ V` without materializing the `L x L` score matrix.

Each program owns a `BM` rows of `Q` and produces those rows of `O`. Inside, loop over `K,V` in `BN`-sized chunks, applying the **online softmax** recurrence to a running output accumulator.

The recurrence for the output (Dao et al., 2022) — for each new chunk `(K_b, V_b)`:

```
  s          = Q @ K_b.T / sqrt(D)         # (BM, BN)
  m_new      = max(m, rowmax(s))            # (BM,)
  α          = exp(m - m_new)               # (BM,)         rescale factor
  p          = exp(s - m_new[:,None])       # (BM, BN)
  l          = l * α + rowsum(p)            # (BM,)
  O_acc      = O_acc * α[:,None] + p @ V_b  # (BM, D)
  m          = m_new
finalize: O = O_acc / l[:,None]
```

🧠 **Fun fact**: the *only* difference between this and the original FlashAttention paper is (a) we kept everything fp32, (b) no causal mask, (c) no batching, (d) we don't recompute on the backward pass. Otherwise — this is it.

In [ ]:
@triton.jit
def k_flash_attn_lite(Q,K,V,O, L, D, sq_l,sq_d, sk_l,sk_d, sv_l,sv_d, so_l,so_d,
                      BM: tl.constexpr, BN: tl.constexpr, D_CONST: tl.constexpr):
    pid = tl.program_id(0)
    rm  = pid*BM + tl.arange(0, BM)               # query rows owned
    rd  = tl.arange(0, D_CONST)                   # the D axis (fits in one tile)
    q   = tl.load(Q + rm[:,None]*sq_l + rd[None,:]*sq_d,
                  mask=(rm<L)[:,None], other=0.0)  # (BM, D)
    scale = 1.0 / tl.sqrt(tl.full((), D, tl.float32))

    m_i = tl.full((BM,), -float('inf'), tl.float32)
    l_i = tl.zeros((BM,), tl.float32)
    o_i = tl.zeros((BM, D_CONST), tl.float32)

    for n0 in range(0, L, BN):
        rn = n0 + tl.arange(0, BN); mn = rn < L
        k = tl.load(K + rn[:,None]*sk_l + rd[None,:]*sk_d, mask=mn[:,None], other=0.0)  # (BN,D)
        v = tl.load(V + rn[:,None]*sv_l + rd[None,:]*sv_d, mask=mn[:,None], other=0.0)  # (BN,D)
        # TODO 1: s = tl.dot(q, tl.trans(k)) * scale                # (BM,BN)
        # TODO 2: mask invalid columns:  s = tl.where(mn[None,:], s, -float('inf'))
        # TODO 3: m_new = tl.maximum(m_i, tl.max(s, 1))
        # TODO 4: alpha = tl.exp(m_i - m_new)
        # TODO 5: p     = tl.exp(s - m_new[:, None])
        # TODO 6: l_i   = l_i * alpha + tl.sum(p, 1)
        # TODO 7: o_i   = o_i * alpha[:, None] + tl.dot(p.to(v.dtype), v)
        # TODO 8: m_i   = m_new
        pass

    o_i = o_i / l_i[:, None]
    tl.store(O + rm[:,None]*so_l + rd[None,:]*so_d, o_i, mask=(rm<L)[:,None])

def test_p24():
    L,D = 128, 64
    Q=torch.randn(L,D,device=DEVICE); K=torch.randn(L,D,device=DEVICE); V=torch.randn(L,D,device=DEVICE)
    O=torch.empty(L,D,device=DEVICE)
    BM=32; BN=32
    k_flash_attn_lite[(triton.cdiv(L,BM),)](Q,K,V,O, L, D,
        Q.stride(0),Q.stride(1), K.stride(0),K.stride(1),
        V.stride(0),V.stride(1), O.stride(0),O.stride(1),
        BM=BM, BN=BN, D_CONST=D)
    ref = torch.softmax(Q @ K.T / math.sqrt(D), dim=-1) @ V
    check(O, ref, atol=1e-3, rtol=1e-3, name='P24 flash_attn_lite')
test_p24()


## Puzzle 25 — Atomic histogram
Given `x` of length `N` with integer values in `[0, BINS)`, compute counts per bin using `tl.atomic_add`. This is also how you'd implement a scatter-gather embedding gradient.

In [ ]:
@triton.jit
def k_hist(x_ptr,hist_ptr,N,BINS,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    # TODO: load idx = tl.load(x_ptr+offs, mask, other=0).to(tl.int32)
    # TODO: tl.atomic_add(hist_ptr + idx, 1, mask=mask)
    pass

def test_p25():
    N=50_000; BINS=64
    x=torch.randint(0,BINS,(N,),device=DEVICE,dtype=torch.int32)
    hist=torch.zeros(BINS,device=DEVICE,dtype=torch.int32)
    k_hist[(triton.cdiv(N,1024),)](x,hist,N,BINS,BLOCK=1024)
    ref = torch.bincount(x.long(),minlength=BINS).to(torch.int32)
    check(hist.float(), ref.float(), name='P25 histogram')
test_p25()


## Puzzle 26 — Swizzled program ordering for L2 reuse (bonus, no test)
When you launch a `(GM, GN)` matmul grid, the *order* the SM scheduler picks programs in matters. The default row-major sweep means by the time you come back to `A[0,:]` for column `pn=GN-1`, it's evicted from L2.

**Trick**: re-map `(pm, pn)` into super-blocks of `GROUP_M` rows so neighboring programs share rows of `A` *and* columns of `B`. This is the famous "L2 swizzle" from Triton's tutorial.

```python
pid = tl.program_id(0)
num_pid_m = tl.cdiv(M, BM); num_pid_n = tl.cdiv(N, BN)
num_pid_in_group = GROUP_M * num_pid_n
group_id    = pid // num_pid_in_group
first_pid_m = group_id * GROUP_M
group_size_m = min(num_pid_m - first_pid_m, GROUP_M)
pm = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
pn = (pid % num_pid_in_group) // group_size_m
```

On a 4096³ matmul this is typically a **+15–30% speedup** for free. Try plugging this into P20 and benchmarking.

🧠 **Fun fact**: this isn't Triton-specific — it's the same trick CUTLASS uses ("thread-block swizzling"). It's purely about which SMs touch which memory first.

## Benchmarking

Triton ships `triton.testing.do_bench` — a properly-warmed timer that handles the GPU cache, ms precision, etc. Here's how you'd compare your matmul vs `torch.matmul`:

```python
import triton.testing as tt
M=N=K=2048
A=torch.randn(M,K,device='cuda',dtype=torch.float16)
B=torch.randn(K,N,device='cuda',dtype=torch.float16)
C=torch.empty(M,N,device='cuda',dtype=torch.float16)

ms_torch = tt.do_bench(lambda: A @ B)
ms_yours = tt.do_bench(lambda: your_matmul(A,B,C))
tflops = lambda ms: 2*M*N*K * 1e-12 / (ms*1e-3)
print(f'torch: {ms_torch:.3f} ms = {tflops(ms_torch):.1f} TFLOPS')
print(f'yours: {ms_yours:.3f} ms = {tflops(ms_yours):.1f} TFLOPS')
```

Goals:
- P19/P20 fp32 matmul: 30–50% of cuBLAS without autotuning.
- With autotuning (`@triton.autotune` over `BM,BN,BK,num_warps,num_stages`): 80–95%.
- P24 flash-attn-lite vs `F.scaled_dot_product_attention`: probably 30–40% of FA2 — you'd need batched, fp16, causal mask, and `num_stages` pipelining to close the gap.

## More fun facts to keep in your back pocket

1. **`num_warps` and `num_stages`** are tunable knobs on every `@triton.jit` call. `num_warps=4` means 128 threads per program; `num_stages=3` means triple-buffered software pipelining of HBM→shared. Try `num_stages=2` for fp32, `3-4` for fp16/bf16.
2. **`tl.dot` requires inputs of shape `(M,K)×(K,N)` with `M,N,K` each ≥ 16** (or 8 on Hopper). Smaller tiles fall back to non-tensor-core paths and become slow.
3. **`tl.load(..., eviction_policy='evict_last')`** hints the L2 cache that this line will be reused — useful for the inner operand of matmul. `'evict_first'` is the opposite, good for streaming.
4. **Triton on AMD (MI300, MI250)** uses MFMA instead of MMA, but the same Python kernel runs. Tile sizes change for best perf, but the API is identical.
5. **Debug print**: `tl.device_print('x', x)` works inside kernels (GPU side) — invaluable for figuring out what your tile actually contains. Also: `TRITON_INTERPRET=1` runs the kernel on CPU and `print()` becomes real Python `print`.
6. **The Triton compiler is MLIR**. You can dump every stage with `TRITON_CACHE_DIR=/tmp/tc python ...` then inspect `~/.triton/cache/` → you'll find `.ttir` (Triton IR), `.ttgir` (Triton GPU IR, with layouts), `.llir` (LLVM), `.ptx`, `.cubin`. Reading the `.ttgir` is where you'll find out *why* your kernel uses shared memory.
7. **Why blocks are powers of 2**: the layout system represents tile shapes in `2^k * 2^k` blocks per warp; non-pow2 forces padding, which wastes registers and shared memory.
8. **Register pressure**: each program's tile lives in registers. A `128x128` fp32 tile = 16,384 floats = 65 KB > the per-thread register file. The compiler will spill to local memory (DRAM!) and your kernel tanks. Watch `ncu --metrics launch__registers_per_thread`.
9. **`tl.static_range` vs `range`**: `static_range` unrolls at compile time (use for small `K` like a conv kernel size). `range` keeps a real loop. Mis-using `static_range` for `K=4096` will OOM your compiler.
10. **Autotuning is your friend**: wrap kernels with `@triton.autotune(configs=[...], key=['M','N','K'])` and Triton will pick the best `(BM,BN,BK,num_warps,num_stages)` for each unique shape on first call.

## Where to go next

- **FlashAttention-2 in Triton**: extend P24 with batching, multi-head, causal mask, fp16, and a backward pass. The reference is `flash_attn/flash_attn_triton.py` and `triton/python/tutorials/06-fused-attention.py`.
- **Persistent kernels**: launch exactly `num_SMs` programs, each loops over multiple tiles internally. Removes launch overhead for tiny shapes (great for decoding).
- **Block-sparse attention**: same as P24 but skip `(BM, BN)` blocks listed in a mask. The basis for sliding-window / longformer / dynamic sparse attention.
- **Mixed-precision quantized matmul**: load `int4` weights, dequantize in registers, do `tl.dot` in fp16. The pattern behind GPTQ/AWQ Triton kernels.
- **Triton + CUDA Graphs**: capture once, replay 10⁵ times. Combined with persistent kernels this is the LLM-inference low-latency stack.

# 🔒 SOLUTIONS — stop scrolling unless you tried!


In [ ]:
# P22
@triton.jit
def sol_p22(x_ptr,w_ptr,y_ptr,N,K,BLOCK: tl.constexpr,KSIZE: tl.constexpr):
    pid=tl.program_id(0); rn = pid*BLOCK + tl.arange(0,BLOCK)
    acc = tl.zeros((BLOCK,), dtype=tl.float32)
    for k in tl.static_range(KSIZE):
        x = tl.load(x_ptr + rn + k, mask=(rn+k)<N, other=0.0)
        wk = tl.load(w_ptr + k)
        acc += x * wk
    tl.store(y_ptr+rn, acc, mask=rn<(N-KSIZE+1))

# P23
@triton.jit
def sol_p23(x_ptr,y_ptr,N,BLOCK: tl.constexpr):
    m = -float('inf'); l = 0.0
    for k in range(0, N, BLOCK):
        rk = k + tl.arange(0, BLOCK); mask = rk < N
        x = tl.load(x_ptr+rk, mask=mask, other=-float('inf'))
        m_new = tl.maximum(m, tl.max(x,0))
        l = l * tl.exp(m - m_new) + tl.sum(tl.exp(x - m_new), 0)
        m = m_new
    for k in range(0, N, BLOCK):
        rk = k + tl.arange(0, BLOCK); mask = rk < N
        x = tl.load(x_ptr+rk, mask=mask, other=-float('inf'))
        y = tl.exp(x - m) / l
        tl.store(y_ptr+rk, y, mask=mask)

# P24
@triton.jit
def sol_p24(Q,K,V,O, L, D, sq_l,sq_d, sk_l,sk_d, sv_l,sv_d, so_l,so_d,
            BM: tl.constexpr, BN: tl.constexpr, D_CONST: tl.constexpr):
    pid = tl.program_id(0)
    rm  = pid*BM + tl.arange(0, BM)
    rd  = tl.arange(0, D_CONST)
    q   = tl.load(Q + rm[:,None]*sq_l + rd[None,:]*sq_d, mask=(rm<L)[:,None], other=0.0)
    scale = 1.0 / tl.sqrt(tl.full((), D, tl.float32))
    m_i = tl.full((BM,), -float('inf'), tl.float32)
    l_i = tl.zeros((BM,), tl.float32)
    o_i = tl.zeros((BM, D_CONST), tl.float32)
    for n0 in range(0, L, BN):
        rn = n0 + tl.arange(0, BN); mn = rn < L
        k = tl.load(K + rn[:,None]*sk_l + rd[None,:]*sk_d, mask=mn[:,None], other=0.0)
        v = tl.load(V + rn[:,None]*sv_l + rd[None,:]*sv_d, mask=mn[:,None], other=0.0)
        s = tl.dot(q, tl.trans(k)) * scale
        s = tl.where(mn[None,:], s, -float('inf'))
        m_new = tl.maximum(m_i, tl.max(s,1))
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(s - m_new[:,None])
        l_i = l_i*alpha + tl.sum(p,1)
        o_i = o_i*alpha[:,None] + tl.dot(p.to(v.dtype), v)
        m_i = m_new
    o_i = o_i / l_i[:,None]
    tl.store(O + rm[:,None]*so_l + rd[None,:]*so_d, o_i, mask=(rm<L)[:,None])

# P25
@triton.jit
def sol_p25(x_ptr,hist_ptr,N,BINS,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    idx = tl.load(x_ptr+offs, mask=mask, other=0).to(tl.int32)
    tl.atomic_add(hist_ptr + idx, 1, mask=mask)

print('All 25 solution kernels defined. Compare with your TODO bodies.')

print('Solutions for puzzles 22–26 defined.')


## Closing thought

If P23 and P24 are passing for you: you have written, by hand, the **core algorithmic kernel** that powers modern LLM serving. The pieces that make production FlashAttention (FA2/FA3) faster than this are *engineering* (fp16/bf16, pipelined `num_stages`, warp specialization on Hopper, persistent kernels, autotuning) — not algorithmic. You now know the algorithm.

Debug loop when something fails:
1. `os.environ['TRITON_INTERPRET']='1'` and re-import — runs your kernel on CPU and lets you `print` inside.
2. Drop the problem size to tiny (`N=8, BLOCK=4`) and trace by hand.
3. `tl.device_print('name', value)` from inside the kernel for GPU-side prints.
